In [1]:
from mesh.core import *
from problems import get_problem

from pymoo.core.problem import Problem
from pymoo.optimize import minimize
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PolynomialMutation
from pymoo.indicators.hv import HV

from hyperopt import fmin, tpe, hp

## Experiment configuration:

In [ ]:
# Algorithm configuration
# Define the objective/fitness function
experiment_name = 'dtlz1'
objective_dim = 3 # Number of objectives
position_dim = 10 # Design space dimension
func, position_min_value, position_max_value = get_problem(experiment_name, n_var=position_dim, n_obj=objective_dim)
population_size = 100 # Population size
max_fit_eval = 15000 # Maximum number of fitness evaluations
random_state = None

# Fine tuning configuration
max_experiments = 100 # Maximum number of evaluations for hyperopt
ref_point = np.array([100] * objective_dim) # Reference point for hypervolume calculation
indicator = HV(ref_point=ref_point) # Define the indicator to calculate the volume

## Create the hyperparameter optimizer for MESH:

In [6]:
num_final_solutions = population_size # Number of final solutions
memory_size = population_size # Maximum number of particles in memory

global_best_attribution_type = 0 # 0 -> E1 | 1 -> E2 | 2 -> E3 | 3 -> E4
dm_pool_type = 1 # 0 -> V1 | 1 -> V2 | 2 -> V3
dm_operation_type = 0 # 0 -> DE\rand\1\Bin (D1) | 1 -> DE\rand\2\Bin (D2) | 2 -> DE/Best/1/Bin (D3) | 3 -> DE/Current-to-best/1/Bin (D4) | 4 -> DE/Current-to-rand/1/Bin (D5)

# Function used by hyperopt to optimize MESH for a function by the hypervolume
def optimize_hyperparameter_by_hypervolume_mesh(function, hyperparameters, indicator):
    # Get the hyperparameters from the optimizer
    communication_probability = hyperparameters["communication_probability"]
    mutation_rate = hyperparameters["mutation_rate"]
    personal_guide_array_size = hyperparameters["personal_guide_array_size"]
    # Configure the MESH to the hyperparameters
    mesh_params = MeshParameters(objective_dim,
                                 position_dim, position_min_value, position_max_value, 
                                 population_size, memory_size=memory_size,
                                 global_best_attribution_type=global_best_attribution_type,
                                 dm_pool_type=dm_pool_type,
                                 dm_operation_type=dm_operation_type,
                                 communication_probability=communication_probability, mutation_rate=mutation_rate,
                                 max_gen=None, max_fit_eval=max_fit_eval,
                                 max_personal_guides=personal_guide_array_size,
                                 random_state=random_state)
    mesh = Mesh(mesh_params, function)
    # Run MESH
    mesh.run()
    # Get the results from MESH
    Pos, Fit = mesh.get_results()
    # Calculate the hypervolume
    hypervolume = indicator(Fit)
    return -hypervolume

## Apply the fine tuning to the MESH:

In [7]:
# Definition of design space of hyperparameters
hyperperameter_space_mesh = {
    "communication_probability": hp.uniform("communication_probability", 0, 1),
    "mutation_rate": hp.uniform("mutation_rate", 0, 1),
    "personal_guide_array_size": hp.choice("personal_guide_array_size", [1, 2, 3])
}

best = fmin(lambda params: optimize_hyperparameter_by_hypervolume_mesh(func, params, indicator), hyperperameter_space_mesh, algo=tpe.suggest, max_evals=max_experiments, verbose=0)
for hyperparam, value in best.items():
    print(f"{hyperparam} = {value}")

communication_probability = 0.6485983400773323
mutation_rate = 0.8772950372354967
personal_guide_array_size = 0


## Apply the fine tuning to the NSGA2:

In [10]:
class MyProblem(Problem):
    def __init__(self, n_var, xl, xu):
        super().__init__(n_var=n_var, n_obj=objective_dim, n_constr=0, xl=xl, xu=xu)

    def _evaluate(self, X, out, *args, **kwargs):
        out["F"] = np.array([func(x) for x in X])

problem = MyProblem(n_var=position_dim, xl=position_min_value, xu=position_max_value)

# Function used by hyperopt to optimize MESH for a function by the hypervolume
def optimize_hyperparameter_by_hypervolume_mesh(problem, hyperparameters, indicator):
    # Get the hyperparameters from the optimizer
	prob_rec = hyperparameters['recombination_probability']
	eta_rec = hyperparameters['eta_recombination']
	prob_mut = hyperparameters['mutation_probability']
	eta_mut = hyperparameters['eta_mutation']
	crossover = SBX(prob=prob_rec, prob_var=1.0, eta=eta_rec)
	mutation = PolynomialMutation(prob=prob_mut, eta=eta_mut)
    # Configure the NSGA-II algorithm
	nsga2 = NSGA2(
		pop_size=population_size,
		crossover=crossover,
		mutation=mutation,
		eliminate_duplicates=True,
		seed=random_state
	)
	# Run NSGA-II
	res = minimize(problem,
				nsga2,
                ('n_eval', max_fit_eval),
                verbose=False)
    # Calculate the hypervolume
	hypervolume = indicator(res.F)
	return -hypervolume

# Definition of design space of hyperparameters
hyperperameter_space = {
	"recombination_probability": hp.uniform("recombination_probability", 0, 1),
	"eta_recombination": hp.uniform("eta_recombination", 0, 20),
	"mutation_probability": hp.uniform("mutation_probability", 0, 1),
	"eta_mutation": hp.uniform("eta_mutation", 0, 20),
}

# Define the indicator to calculate the volume
indicator = HV(ref_point=ref_point)

best = fmin(lambda params: optimize_hyperparameter_by_hypervolume_mesh(problem, params, indicator), hyperperameter_space, algo=tpe.suggest, max_evals=max_experiments, verbose=0)
for hyperparam, value in best.items():
    print(f"{hyperparam} = {value}")

eta_mutation = 9.711869238187358
eta_recombination = 14.918324901291795
mutation_probability = 0.4169253477713241
recombination_probability = 0.6347067968622627
